# TP : Dashboard d'Analyse et d'Aide à la Décision Métier (2h)
## Cas d'usage : Classification & Calibration Morphologique (Dataset Iris)

### Contexte métier
Vous êtes mandaté par un producteur horticole. L'objectif est de formaliser des règles de tri automatisé basées sur des visualisations rigoureuses et des seuils quantifiés, puis de construire le prototype de dashboard interactif pour les équipes de conditionnement.

---
### Les 4 missions d'analyse :
1. **Mission 1 — Cartographie et matrice globale de corrélation (`pairplot`)** : identification visuelle du pouvoir discriminant.
2. **Mission 2 — Règle de décision n°1 (Isolation de l'espèce *setosa*)** : mise en évidence par histogramme/KDE du seuil parfait sur les pétales.
3. **Mission 3 — Règle de décision n°2 (Arbitrage *versicolor* vs *virginica*)** : analyse bivariée (`scatterplot`) et calcul du ratio d'élancement.
4. **Mission 4 — Rapport de synthèse statistique orienté production** : table récapitulative des seuils métier d'acceptation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df_iris = sns.load_dataset("iris")
print("Dataset Iris chargé avec succès :", df_iris.shape)
df_iris.head(3)

## Mission 1 : Cartographie globale & Détection du pouvoir discriminant (30 min)

L'équipe souhaite savoir d'un coup d'œil quelles variables (sépales ou pétales) portent l'information discriminante.

### Consignes :
1. Générez la matrice complète `sns.pairplot` du DataFrame `df_iris` en séparant les espèces par la couleur (`hue='species'`).
2. Identifiez visuellement les variables où les distributions des espèces ne se chevauchent pas.
3. Calculez la matrice de corrélation de Pearson entre les 4 grandeurs numériques dans `matrice_corr` (utilisez `numeric_only=True`).
4. Déterminez la paire de variables continues ayant la corrélation positive la plus forte.

In [ ]:
# TODO 1 : Affichage du Pairplot (à commenter lors des exécutions batch)
grid = sns.pairplot(df_iris, hue="species", diag_kind="kde")
plt.show()

# TODO 2 : Matrice de corrélation
matrice_corr = ...

# Récupération de la corrélation pétale longueur / pétale largeur
corr_petales = matrice_corr.loc["petal_length", "petal_width"]
print(f"Corrélation Longueur/Largeur des pétales : {corr_petales:.3f}")

In [ ]:
assert isinstance(matrice_corr, pd.DataFrame), "matrice_corr doit être un DataFrame."
assert matrice_corr.shape == (4, 4), "La matrice doit couvrir les 4 variables numériques."
assert corr_petales > 0.90, "La corrélation entre les dimensions des pétales doit être supérieure à 0.90."
print("✅ Mission 1 validée : Les pétales portent la quasi-totalité de la séparabilité !")

## Mission 2 : Règle de décision n°1 — Isolation de *setosa* (30 min)

### Question métier :
*« Peut-on certifier qu'une fleur est un Iris setosa uniquement en mesurant la longueur de son pétale en caisse de tri ? »*

### Consignes :
1. Créez une figure `fig2, ax2 = plt.subplots(figsize=(8, 4))`.
2. Tracez la distribution comparée de `petal_length` avec `sns.histplot(data=df_iris, x='petal_length', hue='species', kde=True, ax=ax2)`.
3. Calculez dans `longueur_max_setosa` la valeur maximale observée pour l'espèce *setosa*.
4. Calculez dans `longueur_min_autres` la valeur minimale observée pour toutes les autres espèces (*versicolor* et *virginica*).
5. Formulez le seuil d'exclusion strict : vérifiez qu'il n'existe aucun chevauchement (`longueur_max_setosa < longueur_min_autres`).

In [ ]:
# TODO : Visualisation de la séparation nette
fig2, ax2 = plt.subplots(figsize=(8, 4))
sns.histplot(data=df_iris, x="petal_length", hue="species", kde=True, ax=ax2)
ax2.set_title("Distribution de la longueur des pétales par espèce")
plt.show()

# TODO : Calcul des métriques de coupure
longueur_max_setosa = ...
longueur_min_autres = ...

print(f"Pétale le plus long chez setosa : {longueur_max_setosa} cm")
print(f"Pétale le plus court chez les autres espèces : {longueur_min_autres} cm")

In [ ]:
assert longueur_max_setosa < 2.5, "Le maximum de setosa ne doit pas excéder 2.5 cm."
assert longueur_min_autres > 2.5, "Le minimum des autres espèces doit être supérieur à 2.5 cm."
assert longueur_max_setosa < longueur_min_autres, "Il ne doit y avoir aucun chevauchement entre setosa et les autres."
print("✅ Mission 2 validée : Règle métier confirmée : si pétale < 2.5 cm -> 100% Setosa !")

## Mission 3 : Règle de décision n°2 — Arbitrage *versicolor* vs *virginica* (35 min)

Ces deux espèces présentent une zone de recouvrement morphologique. L'équipe d'agronomie suggère d'utiliser un indicateur combiné : le **ratio de surface du pétale** (`surface_petale = petal_length * petal_width`).

### Consignes :
1. Ajoutez au DataFrame `df_iris` la colonne calculée `surface_petale` (en cm²).
2. Créez une figure `fig3, ax3 = plt.subplots(figsize=(8, 5))` et tracez un `sns.scatterplot` représentant `petal_length` (axe X) vs `petal_width` (axe Y) avec coloration selon `species`.
3. Pour les deux espèces complexes (*versicolor* et *virginica* uniquement) :
   - Calculez la médiane de la surface du pétale pour *versicolor* dans `mediane_surface_versicolor`.
   - Calculez la médiane de la surface du pétale pour *virginica* dans `mediane_surface_virginica`.
4. Évaluez le seuil empirique de bascule défini à **7.5 cm²** : comptez combien de fleurs seraient mal classées si l'on applique la règle `si surface_petale >= 7.5 -> virginica sinon versicolor` dans `nb_erreurs_classement`.

In [ ]:
# TODO : Calcul de la surface et tracé bivarié
df_iris["surface_petale"] = ...

fig3, ax3 = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df_iris, x="petal_length", y="petal_width", hue="species", style="species", s=70, ax=ax3)
ax3.set_title("Espace morphologique des pétales (Frontière de décision)")
plt.show()

# TODO : Calculs d'aide à la décision
mediane_surface_versicolor = ...
mediane_surface_virginica = ...

# Évaluation du taux d'erreur au seuil 7.5 cm² (sur versicolor et virginica uniquement)
df_complexe = df_iris[df_iris["species"].isin(["versicolor", "virginica"])].copy()
prediction_virginica = df_complexe["surface_petale"] >= 7.5
verite_virginica = df_complexe["species"] == "virginica"
nb_erreurs_classement = (prediction_virginica != verite_virginica).sum()

print(f"Médiane surface versicolor : {mediane_surface_versicolor:.2f} cm²")
print(f"Médiane surface virginica  : {mediane_surface_virginica:.2f} cm²")
print(f"Nombre de fleurs mal étiquetées au seuil 7.5 cm² : {nb_erreurs_classement} / {len(df_complexe)}")

In [ ]:
assert "surface_petale" in df_iris.columns, "La colonne surface_petale doit être présente."
assert mediane_surface_versicolor < mediane_surface_virginica, "La médiane de versicolor doit être inférieure à celle de virginica."
assert nb_erreurs_classement <= 6, f"Le seuil 7.5 cm² ne doit pas générer plus de 6 erreurs sur 100 fleurs (obtenu : {nb_erreurs_classement})."
print("✅ Mission 3 validée : Précision de tri supérieure à 94% obtenue sans machine learning !")

## Mission 4 : Table de spécification technique pour la production (25 min)

Avant de concevoir l'interface Streamlit, vous devez fournir au responsable d'usine un tableau synthétique contenant les tolérances géométriques de chaque variété.

### Consignes :
1. Utilisez `groupby('species')` pour agréger `df_iris` et construire une table récapitulative `df_specifications` contenant :
   - `longueur_moyenne_petale` : moyenne de `petal_length` (arrondie à 2 décimales)
   - `largeur_moyenne_petale` : moyenne de `petal_width` (arrondie à 2 décimales)
   - `borne_min_surface` : minimum de `surface_petale`
   - `borne_max_surface` : maximum de `surface_petale`
2. Sauvegardez cette table de référence au format CSV : `specifications_iris_production.csv`.

In [ ]:
# TODO : Construction de la table de règles d'usine
df_specifications = df_iris.groupby("species").agg(
    longueur_moyenne_petale=("petal_length", lambda x: round(x.mean(), 2)),
    largeur_moyenne_petale=("petal_width", lambda x: round(x.mean(), 2)),
    borne_min_surface=("surface_petale", "min"),
    borne_max_surface=("surface_petale", "max")
)

df_specifications.to_csv("specifications_iris_production.csv")
df_specifications

In [ ]:
assert len(df_specifications) == 3, "La table doit contenir les 3 espèces."
assert "borne_max_surface" in df_specifications.columns, "La colonne borne_max_surface est requise."
assert df_specifications.loc["setosa", "borne_max_surface"] < 1.0, "La surface maximale d'un pétale de setosa ne dépasse pas 1 cm²."
print("🎉 FÉLICITATIONS ! Vous avez formalisé les règles métier nécessaires au dashboard !")